# HumanAI Detect - Insan Metni Augmentasyonu: human_backtranslated (Colab GPU)

**Baglam (bkz. proje notlari, 2026-07-21 -- DAMAGE makalesi, Pangram Labs, arXiv:2501.03437):** Su ana kadar SADECE ai_raw metni back-translation ile humanize ediliyordu. DAMAGE makalesi hem insan hem AI belgesini AYNI humanizer'dan gecirip ikisini de egitime katiyor -- gerekce: modelin 'bu sadece ceviri artigi, AI imzasi degil' ayrimini bir invariance olarak ogrenmesi. Ablation'larinda bu yaklasim TPR'i %96.83 -> %98.26 iyilestirdi (Tablo 8).

**KRITIK -- etiket DEGISMIYOR:** Cikti label='human' olarak kaydedilir, YENI SINIF ACILMIYOR -- sadece human sinifi paraphrase-invariance ile zenginlestiriliyor.

**Kaynak:** `data/interim/human/human.jsonl` (3262 ornek, zaten filtrelenmis production kumesi) -- `data/raw/human/human.jsonl` DEGIL (o eski/fazlalik kayitlar iceriyor).

**Neden Colab:** Backtranslation CPU'da ~64sn/ornek -- 3262 ornek yerelde ~58 saat surer, Colab GPU'da muhtemelen 3-6 saate iner (1000 orneklik onceki ai_humanized_openai topup'u Colab'da 1-2 saat surmustu, bu ~3.3x buyuk).

**ONEMLI -- once kontrol et:** Yerel kod degisikliklerinin (humanizers.py, humanize_human_topup.py, build_features.py grup-fix'i) GitHub'a push edildiginden emin ol.


In [ ]:
# 1. GPU kontrolu
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('UYARI: GPU yok! Runtime -> Change runtime type -> GPU (T4 yeterli) secip tekrar dene.')


In [ ]:
# 2. Repo klonla (guncel kod)
GITHUB_REPO = 'https://github.com/BurakCANKURT/humanai-detect.git'
!git clone {GITHUB_REPO} /content/humanai_detect
%cd /content/humanai_detect
!git log --oneline -3


In [ ]:
# 3. Bagimliliklari kur
!pip install -e '.[dev,colab]' -q
!pip install sacremoses -q


In [ ]:
# 4. .env olustur (API key gerekmez, backtranslate tamamen lokal/ucretsiz)
env_content = """
OPENAI_API_KEY=
ANTHROPIC_API_KEY=
GEMINI_API_KEY=
LLAMA_API_KEY=
LLAMA_ENDPOINT=
""".strip()
with open('.env', 'w') as f:
    f.write(env_content)
print('.env olusturuldu')


In [ ]:
# 5. Kaynak dosyayi yerel makineden yukle: data/interim/human/human.jsonl (ZORUNLU, 3262 ornek)
from pathlib import Path
from google.colab import files

Path('data/interim/human').mkdir(parents=True, exist_ok=True)
Path('data/raw/human_backtranslated').mkdir(parents=True, exist_ok=True)

print("'human.jsonl' dosyasini sec (data/interim/human/human.jsonl, 3262 ornek)")
uploaded = files.upload()
for name in uploaded:
    Path(name).rename('data/interim/human/human.jsonl')

n = sum(1 for _ in open('data/interim/human/human.jsonl', encoding='utf-8'))
print(f'human.jsonl yuklendi: {n} ornek')


In [ ]:
# 5b. (OPSIYONEL) onceki bir Colab oturumundan kalan checkpoint varsa yukle, yoksa atla
from pathlib import Path
from google.colab import files

print("checkpoint varsa 'human_backtranslated.jsonl' sec, yoksa bu hucreyi calistirma")
uploaded = files.upload()
for name in uploaded:
    Path(name).rename('data/raw/human_backtranslated/human_backtranslated.jsonl')


In [ ]:
# 6. Ana isi calistir: 3262 insan ornegini back-translation ile augmente et.
# Checkpoint'li -- oturum koparsa 8. hucredeki dosyayi indirip 5b ile geri yukle, devam eder.
!python scripts/humanize_human_topup.py


In [ ]:
# 7. Sonucu yerel makineye indir
from pathlib import Path
from google.colab import files

src = Path('data/raw/human_backtranslated/human_backtranslated.jsonl')
if src.exists():
    n = sum(1 for _ in open(src, encoding='utf-8'))
    print(f'{n} ornek -> indiriliyor')
    files.download(str(src))
else:
    print('UYARI: dosya bulunamadi.')


## Indirilen Dosyayi Yerlestirme

Indirilen `human_backtranslated.jsonl`'i `data/raw/human_backtranslated/human_backtranslated.jsonl`'e koy. Sonra `colab_preprocess_human_topup.ipynb` ile on isleme adimina gec.
